In [1]:
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

# Specify paths and hyperparameters for quantization
model_path = "/media/guoyue/GM7/model_weights/models/Qwen/qwen2.5-1.5B-Instruct"
quant_path = "/media/guoyue/GM7/model_weights/models/Qwen/qwen2.5-1.5B-Instruct-Quantized"
quant_config = { "zero_point": True, "q_group_size": 128, "w_bit": 4, "version": "GEMM" }

# Load your tokenizer and model with AutoAWQ
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoAWQForCausalLM.from_pretrained(model_path, device_map="auto", safetensors=True)

/media/guoyue/GM7/conda_env_ubuntu/quanti/lib/python3.10/site-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)
/media/guoyue/GM7/conda_env_ubuntu/quanti/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io

In [2]:
data_path = "/media/guoyue/GM7/data_set/minimind/sft_mini_512.jsonl"

In [5]:
from torch.utils.data import Dataset
from datasets import load_dataset

dataset = load_dataset("json", data_files=data_path, split="train")

Generating train split: 1214724 examples [00:03, 400721.59 examples/s]


In [ ]:
dataset.select(range(5))

{'conversations': [{'role': 'user', 'content': '请告诉我在中国古代的“四大发明”是什么？'},
  {'role': 'assistant',
   'content': '中国古代的“四大发明”是指造纸术、印刷术、火药和指南针。这四项发明对世界文明的发展产生了深远的影响：\n\n1. **造纸术**：据史书记载，东汉时期的蔡伦改进了造纸工艺，使得纸张的生产更加便捷、成本更低，质量也更加优良。这一发明极大地促进了文化的传播和保存。\n\n2. **印刷术**：中国古代的印刷术最早可以追溯到唐代的雕版印刷，到了宋代发展出了活字印刷技术。印刷术的发明极大地促进了知识的传播，降低了书籍的成本，对教育和文化的发展起到了重要的推动作用。\n\n3. **火药**：火药最初是在唐代被发现的，最初可能用于医疗或炼金术。到了宋代，人们开始将火药用于军事目的，发明了各种火器。火药的发明改变了战争的面貌，对世界军事技术的发展产生了重大影响。\n\n4. **指南针**：指南针最初在中国被用于风水测量，后来逐渐发展成为航海导航的重要工具。这一发明极大地促进了海上航行技术的发展，对于新航路的开辟和世界地理大发现起到了关键作用。\n\n这四项发明不仅在中国历史上占有重要地位，而且对全世界的科技进步和文明发展都产生了深远的影响。'}]}

In [16]:
data_col = []
for data in dataset:
    conversations = data["conversations"]
    data_col.append(tokenizer.apply_chat_template(conversations, max_length=512, tokenize=False, add_generation_prompt=False))

In [17]:
len(data_col)

1214724

In [18]:
model.quantize(tokenizer, quant_config=quant_config, calib_data=data_col)

AWQ: 100%|██████████| 28/28 [02:28<00:00,  5.29s/it]


In [ ]:
model.save_quantized(quant_path, safetensors=True, shard_size="4GB")
tokenizer.save_pretrained(quant_path)

('/media/guoyue/GM7/model_weights/models/Qwen/qwen2.5-1.5B-Instruct-Quantized/tokenizer_config.json',
 '/media/guoyue/GM7/model_weights/models/Qwen/qwen2.5-1.5B-Instruct-Quantized/special_tokens_map.json',
 '/media/guoyue/GM7/model_weights/models/Qwen/qwen2.5-1.5B-Instruct-Quantized/vocab.json',
 '/media/guoyue/GM7/model_weights/models/Qwen/qwen2.5-1.5B-Instruct-Quantized/merges.txt',
 '/media/guoyue/GM7/model_weights/models/Qwen/qwen2.5-1.5B-Instruct-Quantized/added_tokens.json',
 '/media/guoyue/GM7/model_weights/models/Qwen/qwen2.5-1.5B-Instruct-Quantized/tokenizer.json')

: 